In [1]:
import pandas as pd
import numpy as np

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    average_precision_score,
    classification_report
)

import os
from pathlib import Path

from tqdm import tqdm

In [2]:
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

X_TRAIN_PATH = '/kaggle/input/datasets/jiahao1510/network-traffic-classification/X_train.npy'
X_VAL_PATH = '/kaggle/input/datasets/jiahao1510/network-traffic-classification/X_val.npy'
X_TEST_PATH = '/kaggle/input/datasets/jiahao1510/network-traffic-classification/X_test.npy'

Y_TRAIN_PATH = '/kaggle/input/datasets/jiahao1510/network-traffic-classification/y_train.npy'
Y_VAL_PATH = '/kaggle/input/datasets/jiahao1510/network-traffic-classification/y_val.npy'
Y_TEST_PATH = '/kaggle/input/datasets/jiahao1510/network-traffic-classification/y_test.npy'

In [3]:
X_train = np.load(X_TRAIN_PATH)
X_val = np.load(X_VAL_PATH)
X_test = np.load(X_TEST_PATH)

y_train = np.load(Y_TRAIN_PATH)
y_val = np.load(Y_VAL_PATH)
y_test = np.load(Y_TEST_PATH)

In [4]:
X_train = torch.tensor(np.transpose(X_train, (0, 2, 1)), dtype=torch.float32)
X_val = torch.tensor(np.transpose(X_val, (0, 2, 1)), dtype=torch.float32)
X_test = torch.tensor(np.transpose(X_test, (0, 2, 1)), dtype=torch.float32)

y_train = torch.tensor(np.argmax(y_train, axis=1), dtype=torch.long)
y_val   = torch.tensor(np.argmax(y_val, axis=1), dtype=torch.long)
y_test  = torch.tensor(np.argmax(y_test, axis=1), dtype=torch.long)

In [5]:
# dataset definition
class SequenceDataset(Dataset):

    def __init__(self, X: torch.tensor, y: torch.tensor):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]

In [6]:
train_dataset = SequenceDataset(X_train, y_train)
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=64)

val_dataset = SequenceDataset(X_val, y_val)
val_dataloader = DataLoader(val_dataset, shuffle=False, batch_size=64)

test_dataset = SequenceDataset(X_test, y_test)
test_dataloader = DataLoader(test_dataset, shuffle=False, batch_size=64)

In [7]:
# model definition

class CNNModel(nn.Module):
    def __init__(self, input_features: int, num_classes: int, timesteps: int):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels=input_features,
            out_channels=128,
            kernel_size=3,
            stride=1,
            padding='same'
        )

        self.shortcut = nn.Conv1d(128, 256, kernel_size=1)

        self.conv2 = nn.Conv1d(
            in_channels=128,
            out_channels=256,
            kernel_size=3,
            stride=1,
            padding='same'
        )

        self.flatten = nn.Flatten(start_dim=1)

        # self.conv3 = nn.Conv1d(
        #     in_channels=256,
        #     out_channels=512,
        #     kernel_size=3,
        #     stride=1,
        #     padding='same'
        # )

        self.classifier = nn.Sequential(
            nn.Linear(256 * timesteps, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

        self._init_weights()

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        identity = x

        x = self.conv2(x)
        x = F.relu(x)
        x = x + self.shortcut(identity)

        x = self.flatten(x)
        x = self.classifier(x)
        return x

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)


In [8]:
device = torch.device('cuda') if torch.cuda.is_available() else 'cpu'

model = CNNModel(
    input_features=59,
    num_classes=5,
    timesteps=5
)

model.to(device)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train.numpy()),
    y=y_train.numpy()
)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

loss_fn = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [9]:
def train(model, dataloader, loss_fn, optimizer, device, num_classes=5):
    model.train()
    train_loss = 0
    all_probs, all_preds, all_labels = [], [], []

    for x, y in dataloader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        logits = model(x)
        loss = loss_fn(logits, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        # prob and pred for each batch
        all_probs.append(torch.softmax(logits, dim=1).detach().cpu().numpy())
        all_preds.append(logits.argmax(dim=1).detach().cpu().numpy())
        all_labels.append(y.cpu().numpy())

    # flatten into 1 array
    probs = np.concatenate(all_probs)
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)

    metrics = compute_metrics(probs, preds, labels, num_classes)
    metrics["loss"] = train_loss / len(dataloader)

    return metrics

def compute_metrics(probs, preds, labels, num_classes=5):
    # multiclass need to use 1 hot
    labels_onehot = np.eye(num_classes)[labels]

    return {
        "f1": f1_score(labels, preds, average="macro", zero_division=0),
        "precision": precision_score(labels, preds, average="macro", zero_division=0),
        "recall": recall_score(labels, preds, average="macro", zero_division=0),
        "pr_auc": average_precision_score(labels_onehot, probs, average="macro"),
        "report": classification_report(
            labels,
            preds,
            target_names=["Backdoor", "CommInj", "DoS", "Reconn", "normal"],
            digits=4,
            zero_division=0
        )
    }

def evaluate(model, dataloader, loss_fn, device, num_classes=5):
    model.eval()
    val_loss = 0
    all_probs, all_preds, all_labels = [], [], []

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            logits = model(x)
            val_loss += loss_fn(logits, y).item()

            # prob and pred for each batch
            all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
            all_preds.append(logits.argmax(dim=1).cpu().numpy())
            all_labels.append(y.cpu().numpy())

    # flatten into 1 array
    probs = np.concatenate(all_probs)
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)

    metrics = compute_metrics(probs, preds, labels, num_classes)
    metrics["loss"] = val_loss / len(dataloader)

    return metrics

In [10]:
EPOCHS = 50
PATIENCE = 5

best_pr_auc = 0.0
patience_left = PATIENCE
ckpt_path = "./models/cnn_best.pt"
os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

pbar = tqdm(range(EPOCHS), desc="Training CNN")

for epoch in pbar:
    train_metrics = train(model, train_dataloader, loss_fn, optimizer, device)
    val_metrics = evaluate(model, val_dataloader, loss_fn, device)

    pbar.set_postfix(
        train_loss=f"{train_metrics['loss']:.4f}",
        train_f1=f"{train_metrics['f1']:.4f}",
        train_pr_auc=f"{train_metrics['pr_auc']:.4f}",
        val_loss=f"{val_metrics['loss']:.4f}",
        val_f1=f"{val_metrics['f1']:.4f}",
        val_pr_auc=f"{val_metrics['pr_auc']:.4f}"
    )

    if val_metrics["pr_auc"] > best_pr_auc:
        best_pr_auc = val_metrics["pr_auc"]
        torch.save(model.state_dict(), ckpt_path)
        patience_left = PATIENCE
    else:
        patience_left -= 1

    if patience_left <= 0:
        tqdm.write(f"Early stopping at epoch {epoch + 1}")
        break

Training CNN:  36%|███▌      | 18/50 [02:58<05:17,  9.93s/it, train_f1=0.8835, train_loss=0.0151, train_pr_auc=0.9804, val_f1=0.8846, val_loss=0.0222, val_pr_auc=0.9772]

Early stopping at epoch 19


In [11]:
model.load_state_dict(torch.load(ckpt_path, map_location=device))

test_metrics = evaluate(model, test_dataloader, loss_fn, device)
print(
    f"Test Loss: {test_metrics['loss']:.4f} | "
    f"F1: {test_metrics['f1']:.4f} | "
    f"Precision: {test_metrics['precision']:.4f} | "
    f"Recall: {test_metrics['recall']:.4f} | "
    f"PR-AUC: {test_metrics['pr_auc']:.4f}"
)

print("\nClassification Report:")
print(test_metrics["report"])

Test Loss: 0.0397 | F1: 0.8485 | Precision: 0.8214 | Recall: 0.9492 | PR-AUC: 0.9539

Classification Report:
              precision    recall  f1-score   support

    Backdoor     0.2345    0.8718    0.3696        39
     CommInj     0.9057    0.9412    0.9231        51
         DoS     1.0000    1.0000    1.0000        91
      Reconn     0.9667    0.9355    0.9508        31
      normal     1.0000    0.9977    0.9988     47152

    accuracy                         0.9975     47364
   macro avg     0.8214    0.9492    0.8485     47364
weighted avg     0.9992    0.9975    0.9982     47364

